# 08. Comparisons between Models

In this notebook, we:
1) Load the inference results from two models.
2) Compare the results across:
    - Latency;
    - Syntactical Fidelity;
    - Interpretative Fidelity.

> Note that interpretative fidelity requires that the outputs from the models have been manually labelled using teh evaluation scale.

In [1]:
import pandas as pd
from collections import Counter, defaultdict

from src.core.types import *
from src.core.analysis.utils import exact_match, normalized_edit_distance, action_type_f1

## 1) Loading the Datasets

In [2]:
# Extract the test set
test_split_ids = pd.read_csv("data/inputs/test-split-ids.csv", header=None)
manually_evaluated_ids = pd.read_csv("data/outputs/selected-data-low-reasoning-gpt5.csv")

manual_ids = manually_evaluated_ids['input_id'].tolist()
test_ids = test_split_ids[0].tolist() + manual_ids

print(f"Number of items to evaluate automatically: {len(test_ids)}")
print(f"Number of items to evaluate manually: {len(manual_ids)}")

Number of items to evaluate automatically: 182
Number of items to evaluate manually: 50


In [3]:
# Utility for the comparison

def print_labels_counter(dataset : list[LLMCommandingLabelledDataPoint]):
    labels = []
    for datapoint in dataset:
        if datapoint.selected_for_labelling:
            datapoint.infer_label()
        else:
            datapoint.label = None
        labels.append(datapoint.label)
    counter = Counter(labels)
    for label in DataPointLabel:
        print(f"{label}: {counter[label]:3d}", end="\t")
    print()

def get_bucket(metric: float) -> int:
    if metric < 0.33:
        return 0   # low
    elif metric < 0.66:
        return 1   # medium
    else:
        return 2   # high

In [89]:
teacher_path = "data/outputs/all-data-gpt5.csv"
student_path = "data/outputs/all-data-qwen-1.5b-trained.csv"

In [90]:
teacher_df = pd.read_csv(teacher_path)
teacher_entries = [LLMCommandingLabelledDataPoint(**row) for row in teacher_df.to_dict(orient="records")]
teacher_test = sorted([entry for entry in teacher_entries if entry.input_id in test_ids], key=lambda x: x.input_id)

student_df = pd.read_csv(student_path)
student_entries = [LLMCommandingLabelledDataPoint(**row) for row in student_df.to_dict(orient="records")]
student_test = sorted([entry for entry in student_entries if entry.input_id in test_ids], key=lambda x: x.input_id)

In [91]:
student_df

,input_id,game_state,command,command_intent,command_explicitness,command_atomicity,command_contextuality,game_actions,latency,reason_if_failed,...,action_full_correct,action_unnecessary,action_imprecise_sequentiality,action_imprecise_parameters,action_harming_sequentiality,action_harming_parameters,action_missing,action_harming,action_wrong_syntax,label
0,state-233-p0-uc0,AIMED_AT:\r\n type: Wall\r\n distance: 330.8...,Go hit that switch on the wall,Approach the interactable wall and activate th...,0.85,0.40,0.85,SPRINT 0.0 330.86\r\nINTERACT,0.228124,NaN,...,0,0,0,0,0,0,0,0,0,NaN
1,state-233-p0-uc1,AIMED_AT:\r\n type: Wall\r\n distance: 330.8...,Look around for any new openings,Survey the surroundings for newly revealed pat...,0.60,0.50,0.90,ROTATE 360 0,0.131139,NaN,...,0,0,0,0,0,0,0,0,0,NaN
2,state-233-p0-uc2,AIMED_AT:\r\n type: Wall\r\n distance: 330.8...,Keep the pistol ready and move forward,Advance cautiously while staying prepared for ...,0.70,0.50,0.70,MOVE 0.0 330.86,0.153828,NaN,...,0,0,0,0,0,0,0,0,0,NaN
3,state-233-p1-uc0,AIMED_AT:\r\n type: Wall\r\n distance: 330.8...,Go press that switch ahead,Move to the interactable wall in front and act...,0.85,0.40,0.85,SPRINT 0.0 330.86\r\nINTERACT,0.175759,NaN,...,0,0,0,0,0,0,0,0,0,NaN
4,state-233-p1-uc1,AIMED_AT:\r\n type: Wall\r\n distance: 330.8...,Hit the use key on that wall,Trigger interaction with the nearby interactab...,0.90,0.85,0.70,INTERACT,0.073179,NaN,...,0,0,0,0,0,0,0,0,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2867,state-48886-p1-uc2,AIMED_AT:\r\n type: Wall\r\n distance: 231.6...,Keep me away from that monster,Avoid close contact with the visible DoomImp e...,0.30,0.50,0.95,ROTATE_TO_TARGET MONSTER_0\r\nSPRINT 0.0 -200.0,0.214781,NaN,...,2,0,0,0,0,0,0,0,0,NaN
2868,state-57055-p1-uc2,AIMED_AT:\r\n type: Monster\r\n distance: 51...,Back up and keep firing at them,Create distance from the swarm while continuin...,0.70,0.30,0.90,ASYNC FIRE 3.0\r\nSPRINT 0 -200,0.237004,NaN,...,2,0,0,0,0,0,0,0,0,NaN
2869,state-57139-p0-uc0,AIMED_AT:\r\n type: Wall\r\n distance: 686.1...,Line up and shoot the closest lost soul,Quickly focus aim on the nearest flying enemy ...,0.78,0.50,0.90,ROTATE_TO_TARGET MONSTER_4\r\nFIRE 1.0,0.207900,NaN,...,1,0,0,1,0,0,0,0,0,NaN
2870,state-57195-p3-uc0,AIMED_AT:\r\n type: Wall\r\n distance: 813.7...,Focus fire on the closest lost soul,Quickly kill the nearest charging lost soul th...,0.85,0.50,0.90,ROTATE_TO_TARGET MONSTER_3\r\nFIRE_SHOTS 5,0.233833,NaN,...,2,0,0,0,0,0,0,0,0,NaN


## 2) Running the Comparisons

In [92]:
print("Teacher labels:")
print_labels_counter(teacher_test)

Teacher labels:
FC:  42	CNO:   6	OW:   0	CW:   2	SW:   0	


In [93]:
print("Student labels:")
print_labels_counter(student_test)

Student labels:
FC:  38	CNO:  10	OW:   0	CW:   2	SW:   0	


In [94]:
outputs_df = []
for teacher, student in zip(teacher_test, student_test):
    assert teacher.input_id == student.input_id

    # Compute EM and Edit Distance
    student.exact_match = exact_match(teacher.game_actions, student.game_actions)
    student.edit_distance = normalized_edit_distance(teacher.game_actions, student.game_actions)

    # Compute F1 score between Game Actions
    teacher_actions = [line.split()[0] for line in teacher.game_actions.replace("ASYNC", "").splitlines() if line.split()]
    student_actions = [line.split()[0] for line in student.game_actions.replace("ASYNC", "").splitlines() if line.split()]
    student.action_type_f1 = action_type_f1(teacher_actions, student_actions)

    outputs_df.append({
        "id": teacher.input_id,
        "command": teacher.command,
        "teacher": teacher.game_actions,
        "student": student.game_actions,
        "em": student.exact_match,
        "ed": student.edit_distance,
        "f1": student.action_type_f1,
        "latency": student.latency,
        "label": student.label,
    })

outputs_df = pd.DataFrame(outputs_df)

In [95]:
outputs_df

,id,command,teacher,student,em,ed,f1,latency,label
0,state-10643-p0-uc2,Ignore the imp and focus on progressing,SPRINT 0.0 636.92\r\nINTERACT,ROTATE_TO_TARGET MONSTER_0\r\nMOVE_TO_TARGET M...,False,0.865385,0.000000,0.167190,None
1,state-10643-p1-uc0,Hit that switch on the wall,SPRINT 0.0 636.92\r\nINTERACT,SPRINT 0.0 636.92\r\nINTERACT,True,0.000000,1.000000,0.202592,None
2,state-11021-p0-uc1,Center aim on that imp and fire,ROTATE_TO_TARGET MONSTER_0\r\nFIRE 1.0,ROTATE_TO_TARGET MONSTER_0\r\nFIRE 1.0,True,0.000000,1.000000,0.187625,DataPointLabel.FULLY_CORRECT
3,state-11021-p1-uc2,Focus on the imp after these guys,ROTATE_TO_TARGET MONSTER_1\r\nFIRE_SHOTS 1\r\n...,ROTATE_TO_TARGET MONSTER_0\r\nFIRE_SHOTS 2,False,0.645455,0.571429,0.159620,None
4,state-11066-p1-uc0,Line up on that imp and fire,ROTATE_TO_TARGET MONSTER_0\r\nFIRE 0.5,ROTATE_TO_TARGET MONSTER_0\r\nFIRE 1.0,False,0.055556,1.000000,0.182224,DataPointLabel.FULLY_CORRECT
...,...,...,...,...,...,...,...,...,...
177,state-8567-p0-uc0,Switch to the shotgun and blast the front guy,SELECT Shotgun\r\nROTATE_TO_TARGET MONSTER_2\r...,SELECT Shotgun\r\nROTATE_TO_TARGET MONSTER_2\r...,False,0.142857,0.666667,0.205678,None
178,state-8567-p0-uc2,Back up while firing at them,SELECT Shotgun\r\nASYNC FIRE 2.0\r\nMOVE 0 -200,SELECT Shotgun\r\nASYNC FIRE 2.0\r\nMOVE 0 -150,False,0.046512,1.000000,0.190103,None
179,state-973-p2-uc0,Shoot that imp ahead,ROTATE_TO_TARGET MONSTER_0\r\nFIRE 1.0,ROTATE_TO_TARGET MONSTER_0\r\nFIRE 1.0,True,0.000000,1.000000,0.168639,None
180,state-991-p2-uc0,Turn toward the closest enemy and attack,FAIL No usable weapon available to attack the ...,ROTATE_TO_TARGET MONSTER_1\r\nMOVE_TO_TARGET M...,False,0.933333,0.000000,0.268357,None


In [96]:
clusters = {dp.cluster_id for dp in student_entries}

for cluster in clusters:
    print(f"Cluster #{cluster}")
    selected_test = [dp for dp in student_entries if dp.cluster_id == cluster]
    print(f"Examples: {selected_test[0].command_intent}, {selected_test[1].command_intent}")
    print_labels_counter(selected_test)
    print("-"*20)

Cluster #0
Examples: Change to a stronger weapon and shoot the spectre, Create distance from the approaching spectre without losing aim
FC:   4	CNO:   0	OW:   0	CW:   1	SW:   0	
--------------------
Cluster #1
Examples: Quickly rotate toward the DoomImp and shoot it with the shotgun, Use the shotgun while dodging to clear the enemy group
FC:   4	CNO:   1	OW:   0	CW:   0	SW:   0	
--------------------
Cluster #2
Examples: Increase distance from enemies while maintaining aim for safer shooting, Maintain distance from both enemies and keep shooting them
FC:   5	CNO:   0	OW:   0	CW:   0	SW:   0	
--------------------
Cluster #3
Examples: Rotate view to line up crosshair with the nearby imp, Fire the pistol repeatedly to kill the visible imp
FC:   4	CNO:   1	OW:   0	CW:   0	SW:   0	
--------------------
Cluster #4
Examples: Dodge potential attacks by circling and firing at the monster, Avoid incoming attacks while continuously shooting the monster
FC:   2	CNO:   3	OW:   0	CW:   0	SW:   0	
---

In [97]:
def print_buckets(metric):
    buckets = defaultdict(list)

    for dp in student_entries:
        b = get_bucket(getattr(dp, metric))
        buckets[b].append(dp)

    bucket_names = {
        0: "Low Explicitness",
        1: "Medium Explicitness",
        2: "High Explicitness"
    }

    for b in sorted(buckets.keys()):
        selected_test = buckets[b]

        # skip empty buckets (important!)
        if len(selected_test) == 0:
            continue

        print(f"Bucket #{b} — {bucket_names[b]}")

        # print example commands safely
        examples = [dp.command_intent for dp in selected_test[:2]]
        print(f"Examples: {', '.join(examples)}")

        print_labels_counter(selected_test)
        print("-"*20)

In [98]:
print_buckets('command_explicitness')

Bucket #0 — Low Explicitness
Examples: Adjust view and stance to scan area for possible enemies, Manage movement and attacks to handle the nearby threat
FC:   2	CNO:   0	OW:   0	CW:   0	SW:   0	
--------------------
Bucket #1 — Medium Explicitness
Examples: Survey the surroundings for newly revealed paths or doors, Explore the surroundings to find a possible exit path
FC:   2	CNO:   0	OW:   0	CW:   1	SW:   0	
--------------------
Bucket #2 — High Explicitness
Examples: Approach the interactable wall and activate the switch or door, Advance cautiously while staying prepared for potential enemies
FC:  34	CNO:  10	OW:   0	CW:   1	SW:   0	
--------------------


In [99]:
print_buckets('command_contextuality')

Bucket #0 — Low Explicitness
Examples: Equip the pistol to prepare for the coming fight, Change weapon to the pistol for efficient shooting
FC:   1	CNO:   0	OW:   0	CW:   0	SW:   0	
--------------------
Bucket #1 — Medium Explicitness
Examples: Advance exploration by moving further along the current path, Advance while following the current wall to explore the area
FC:   1	CNO:   0	OW:   0	CW:   0	SW:   0	
--------------------
Bucket #2 — High Explicitness
Examples: Approach the interactable wall and activate the switch or door, Survey the surroundings for newly revealed paths or doors
FC:  36	CNO:  10	OW:   0	CW:   2	SW:   0	
--------------------


In [100]:
print_buckets('command_atomicity')

Bucket #0 — Low Explicitness
Examples: Adjust view and stance to scan area for possible enemies, Explore the surroundings to find a possible exit path
FC:   7	CNO:   0	OW:   0	CW:   0	SW:   0	
--------------------
Bucket #1 — Medium Explicitness
Examples: Approach the interactable wall and activate the switch or door, Survey the surroundings for newly revealed paths or doors
FC:  27	CNO:   9	OW:   0	CW:   2	SW:   0	
--------------------
Bucket #2 — High Explicitness
Examples: Trigger interaction with the nearby interactable surface, Damage and ideally kill the nearby DoomImp enemy
FC:   4	CNO:   1	OW:   0	CW:   0	SW:   0	
--------------------
